# Tropeiro Intel 4.0 · Investigation Station

**Campaign Intelligence · Relationship Analysis · Attribution Engineering · IOC Decisioning**

Esta é a máquina oficial completa. O notebook expõe todas as etapas da investigação, enquanto a lógica analítica vive no backend `tropeiro/` e o relatório consome os mesmos objetos produzidos aqui.

**Fluxo:** `INGEST → COLLECT → NORMALIZE → EVIDENCE → RELATE → CLUSTER → ATTRIBUTE → DECIDE → PIVOT → REPORT`

> **Modo padrão: PASSIVE.** APIs pagas são opcionais. Ausência de credencial vira `SKIPPED`, nunca quebra o caso. Similaridade gera candidatos, não vereditos. Correlação não é identidade.

In [ ]:
#@title 01) Bootstrap robusto do backend oficial
import os, sys, subprocess, shutil, importlib
from pathlib import Path

REPO = Path("/content/tropeiro-intel")
REPO_URL = "https://github.com/Ridd1kulusC0d3r/tropeiro-intel.git"

def _pip_install(*specs, required=False):
    cmd = [sys.executable, "-m", "pip", "install", "-q", *specs]
    p = subprocess.run(cmd, text=True, capture_output=True)
    if p.returncode != 0:
        print("⚠ pip:", " ".join(specs))
        print((p.stderr or p.stdout or "")[-1800:])
        if required:
            raise RuntimeError("Falha ao instalar dependências obrigatórias.")
        return False
    return True

def ensure_tropeiro(force_sync=False):
    # Colab às vezes deixa /content/tropeiro-intel de uma execução interrompida.
    # Diretório incompleto não é tratado como clone válido.
    if REPO.exists() and not (REPO / ".git").exists():
        print("Limpando clone incompleto:", REPO)
        shutil.rmtree(REPO, ignore_errors=True)

    if not (REPO / ".git").exists():
        print("Clonando backend oficial…")
        p = subprocess.run(
            ["git","clone","--depth","1","-q",REPO_URL,str(REPO)],
            text=True, capture_output=True
        )
        if p.returncode != 0:
            raise RuntimeError("Falha no git clone:\n" + (p.stderr or p.stdout or "")[-1800:])
    elif force_sync:
        print("Sincronizando backend com main…")
        subprocess.run(["git","-C",str(REPO),"fetch","-q","--depth","1","origin","main"], check=False)
        subprocess.run(["git","-C",str(REPO),"reset","-q","--hard","origin/main"], check=False)

    # Importação direta do source tree torna o bootstrap resiliente mesmo se
    # o editable install não atualizar o sys.path da sessão atual.
    if str(REPO) not in sys.path:
        sys.path.insert(0, str(REPO))

    print("Instalando dependências core…")
    editable_ok = _pip_install("-e", str(REPO), required=False)
    if not editable_ok:
        # Fallback explícito. O source tree já está no sys.path.
        _pip_install("tldextract","rapidfuzz","pandas","networkx","stix2", required=True)

    # Extras de interface são best-effort e jamais impedem o backend core.
    _pip_install("ipywidgets", required=False)
    _pip_install("dnstwist", required=False)

    importlib.invalidate_caches()
    try:
        import tropeiro as _tropeiro
    except Exception as e:
        print("REPO existe:", REPO.exists())
        print("tropeiro/__init__.py existe:", (REPO/"tropeiro"/"__init__.py").exists())
        print("sys.path[0:3]:", sys.path[:3])
        raise RuntimeError(
            "Backend foi clonado, mas 'tropeiro' ainda não pôde ser importado. "
            "Use Runtime > Disconnect and delete runtime e execute a célula 01 novamente."
        ) from e
    return _tropeiro

tropeiro = ensure_tropeiro(force_sync=True)
print(f"✓ Tropeiro Intel {tropeiro.__version__} importado de {Path(tropeiro.__file__).resolve()}")
print("✓ Bootstrap concluído. Agora a estação pode continuar.")


In [ ]:
#@title 02) Health check do runtime + autorrecuperação
import sys, platform, shutil, json, os, subprocess, importlib
from pathlib import Path

REPO = Path("/content/tropeiro-intel")

# Se esta célula for executada isoladamente, tenta recuperar o backend.
if "ensure_tropeiro" in globals():
    tropeiro = ensure_tropeiro(force_sync=False)
else:
    if REPO.exists() and str(REPO) not in sys.path:
        sys.path.insert(0, str(REPO))
    try:
        import tropeiro
    except ModuleNotFoundError:
        if REPO.exists() and not (REPO/".git").exists():
            shutil.rmtree(REPO, ignore_errors=True)
        if not (REPO/".git").exists():
            subprocess.run(
                ["git","clone","--depth","1","-q",
                 "https://github.com/Ridd1kulusC0d3r/tropeiro-intel.git",str(REPO)],
                check=True
            )
        if str(REPO) not in sys.path:
            sys.path.insert(0, str(REPO))
        subprocess.run(
            [sys.executable,"-m","pip","install","-q",
             "tldextract","rapidfuzz","pandas","networkx","stix2"],
            check=True
        )
        importlib.invalidate_caches()
        import tropeiro

HEALTH = {
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "tropeiro": tropeiro.__version__,
    "tropeiro_path": str(Path(tropeiro.__file__).resolve()),
    "dnstwist": bool(shutil.which("dnstwist")),
    "repo": (REPO/".git").exists(),
}
print(json.dumps(HEALTH, indent=2, ensure_ascii=False))
if not HEALTH["repo"] or not HEALTH["tropeiro_path"].startswith(str(REPO)):
    raise RuntimeError("Health check falhou: backend não está carregado a partir do clone oficial.")
print("✓ Runtime saudável.")


## A · Case setup, modes & secrets

In [ ]:
#@title 03) Secrets opcionais: Colab Secrets → env → ausente
import os

def _secret(name,default=''):
    try:
        from google.colab import userdata
        v=userdata.get(name)
        if v:return v
    except Exception:pass
    return os.getenv(name,default)
SECRET_NAMES=['URLSCAN_API_KEY','VT_API_KEY','THREATFOX_AUTH_KEY','DNSDUMPSTER_API_KEY','FOFA_API_KEY','CENSYS_PAT','CENSYS_ORG_ID','DOMAINTOOLS_API_KEY','SECURITYTRAILS_API_KEY','WHOISXML_API_KEY']
SECRETS={x:_secret(x) for x in SECRET_NAMES}
print('Secrets disponíveis:',[k for k,v in SECRETS.items() if v] or 'nenhum')

In [ ]:
#@title 04) Configuração principal do caso
CASE_ID='TI-001' #@param {type:'string'}
ANALYST='Analista' #@param {type:'string'}
BRAND='' #@param {type:'string'}
IMPERSONATED_ORG='' #@param {type:'string'}
CAMPAIGN_NOTE='' #@param {type:'string'}
SEEDS='example.com' #@param {type:'string'}
MANUAL_IOCS='' #@param {type:'string'}
LURE_TEXT='' #@param {type:'string'}
MODE='PASSIVE' #@param ['PASSIVE','SAFE_ENRICHMENT','AUTHORIZED_ACTIVE']
PROVIDER_BUDGET='balanced' #@param ['free','balanced','extended']
print('Caso:',CASE_ID,'| modo:',MODE,'| budget:',PROVIDER_BUDGET)

In [ ]:
#@title 05) Feature flags
ENABLE_DNS=True #@param {type:'boolean'}
ENABLE_RDAP=True #@param {type:'boolean'}
ENABLE_CT=True #@param {type:'boolean'}
ENABLE_WAYBACK=True #@param {type:'boolean'}
ENABLE_COMMONCRAWL=False #@param {type:'boolean'}
ENABLE_URLSCAN=True #@param {type:'boolean'}
ENABLE_URLSCAN_DETAILS=True #@param {type:'boolean'}
ENABLE_OTX=True #@param {type:'boolean'}
ENABLE_VT=False #@param {type:'boolean'}
ENABLE_THREATFOX=False #@param {type:'boolean'}
ENABLE_DNSTWIST=True #@param {type:'boolean'}
ENABLE_DNSDUMPSTER=False #@param {type:'boolean'}
ENABLE_FOFA=False #@param {type:'boolean'}
ENABLE_CENSYS=False #@param {type:'boolean'}
ENABLE_DOMAIN_SIMILARITY=True #@param {type:'boolean'}
ENABLE_PASSIVE_TAKEOVER=True #@param {type:'boolean'}
ENABLE_HTTP_PROBE=False #@param {type:'boolean'}
DNSTWIST_MAX=250 #@param {type:'integer'}
URLSCAN_DETAIL_MAX=12 #@param {type:'integer'}
if ENABLE_HTTP_PROBE and MODE!='AUTHORIZED_ACTIVE':
    print('⚠ HTTP probe solicitado fora de AUTHORIZED_ACTIVE: desativado automaticamente.')
    ENABLE_HTTP_PROBE=False

In [ ]:
#@title 06) Imports do backend + workspace
# Esta célula também se autorrecupera caso o runtime tenha perdido o import.
import os, sys, subprocess, importlib, re, json, time, hashlib, base64
from pathlib import Path

_REPO = Path("/content/tropeiro-intel")
if _REPO.exists() and str(_REPO) not in sys.path:
    sys.path.insert(0, str(_REPO))
try:
    import tropeiro
except ModuleNotFoundError:
    if "ensure_tropeiro" in globals():
        tropeiro = ensure_tropeiro(force_sync=False)
    else:
        raise RuntimeError(
            "Backend não carregado. Execute a célula 01 'Bootstrap robusto do backend oficial' "
            "antes desta etapa."
        )

from datetime import datetime, timezone, timedelta
from collections import defaultdict
import pandas as pd
import networkx as nx
from IPython.display import display,HTML,Markdown,IFrame
from tropeiro.models import Observation,now_iso
from tropeiro.utils import extract_iocs,hostname,root_domain
from tropeiro.config import Settings
from tropeiro.collectors import dns,rdap,crtsh,history,urlscan,threatintel
from tropeiro.collectors.dnsdumpster import domain_lookup as dnsdumpster_lookup,normalize as dnsdumpster_normalize
from tropeiro.collectors.fofa import search_domain as fofa_domain
from tropeiro.collectors.censys import search_domain as censys_domain
from tropeiro.similarity import compare_domains,extract_durable_identifiers,passive_web_fingerprint
from tropeiro.correlation.graph import CampaignGraph
from tropeiro.correlation.advanced import source_independence,guarded_components,relationship_strength
from tropeiro.correlation.fingerprint import fingerprint as campaign_fingerprint
from tropeiro.attribution.engine import AttributionEngine
from tropeiro.attribution.operator_fingerprint import operator_fingerprint
from tropeiro.evidence.ledger import build as build_ledger
from tropeiro.intelligence import *
from tropeiro.providers import plan as provider_plan
from tropeiro.timeline import build_timeline
from tropeiro.reporting.rich_html import build_report
from tropeiro.reporting.exporter import export_selected,zip_exports

WORKSPACE=Path(f"/content/tropeiro_{re.sub(r'[^A-Za-z0-9_.-]+','_',CASE_ID)}")
WORKSPACE.mkdir(parents=True,exist_ok=True)
SETTINGS=Settings(case_id=CASE_ID,analyst=ANALYST,brand=BRAND,workspace=WORKSPACE,mode=MODE,provider_budget=PROVIDER_BUDGET,enable_http_probe=ENABLE_HTTP_PROBE)
OBS=[];STATUS=[]
def observe(entity,entity_type,source,value,confidence='observed',notes=''):
    OBS.append(Observation(str(entity),str(entity_type),str(source),str(value),confidence=confidence,notes=str(notes)))
def status(source,state,items=0,error='',seconds=0):
    STATUS.append({'source':source,'status':state,'items':items,'seconds':round(seconds,2),'error':str(error)[:260]})
def safe(source,fn,*args,**kwargs):
    t=time.time()
    try:
        x=fn(*args,**kwargs)
        status(source,'OK',len(x) if hasattr(x,'__len__') else 1,seconds=time.time()-t)
        return x
    except Exception as e:
        status(source,'UNAVAILABLE',0,e,time.time()-t)
        return None

print(f"✓ Backend {tropeiro.__version__} pronto | workspace: {WORKSPACE}")


In [ ]:
#@title 07) Provider Query Planner
flags={'dns':ENABLE_DNS,'rdap':ENABLE_RDAP,'crt.sh':ENABLE_CT,'wayback':ENABLE_WAYBACK,'commoncrawl':ENABLE_COMMONCRAWL,'urlscan':ENABLE_URLSCAN,'otx':ENABLE_OTX,'virustotal':ENABLE_VT,'threatfox':ENABLE_THREATFOX,'dnsdumpster':ENABLE_DNSDUMPSTER,'fofa':ENABLE_FOFA,'censys':ENABLE_CENSYS}
requirements=['dns','registration','ownership','tls','historical_web','web_scan','redirects','threat_intel','relationships','asn']
PROVIDER_PLAN=provider_plan(requirements,SECRETS,flags,PROVIDER_BUDGET)
display(pd.DataFrame(PROVIDER_PLAN)[['name','status','cost_class','matched_capabilities','missing_secrets']])

## B · Ingestion & normalization

In [ ]:
#@title 08) Ingestão multi-IOC
raw='\n'.join([SEEDS,MANUAL_IOCS,LURE_TEXT])
IOCS=extract_iocs(raw)
for item in [x.strip() for x in re.split(r'[\n,;]+',SEEDS+'\n'+MANUAL_IOCS) if x.strip()]:
    if item.startswith(('http://','https://')):
        IOCS.setdefault('url',[]).append(item);IOCS.setdefault('domain',[]).append(hostname(item))
    elif '.' in item:IOCS.setdefault('domain',[]).append(hostname(item))
for k in list(IOCS):IOCS[k]=sorted(set(x for x in IOCS[k] if x))
DOMAINS=sorted(set(IOCS.get('domain',[])));ROOTS=sorted(set(root_domain(x) for x in DOMAINS if root_domain(x)))
for typ,vals in IOCS.items():
    for v in vals:observe(v,typ,'manual/input',v,'provided')
print({k:len(v) for k,v in IOCS.items()});print('Roots:',ROOTS)

In [ ]:
#@title 09) Normalização e case inventory
INVENTORY=[]
for typ,vals in IOCS.items():
    for v in vals:INVENTORY.append({'type':typ,'value':v,'root_domain':root_domain(v) if typ in {'domain','url'} else ''})
display(pd.DataFrame(INVENTORY))

## C · Passive collection

In [ ]:
#@title 10) DNS Intelligence
DNS_DATA={}
if ENABLE_DNS:
    for d in ROOTS:
        DNS_DATA[d]={}
        for rt in ['A','AAAA','CNAME','MX','NS','TXT','CAA']:
            vals=safe(f'dns:{rt}:{d}',dns.query,d,rt) or [];DNS_DATA[d][rt]=vals
            for v in vals:observe(d,'domain',f'dns:{rt}',v)
else:status('dns','SKIPPED')
display(pd.DataFrame([{'domain':d,**{k:len(v) for k,v in x.items()}} for d,x in DNS_DATA.items()]))

In [ ]:
#@title 11) RDAP / ownership público
RDAP_DATA={}
if ENABLE_RDAP:
    for d in ROOTS:
        r=safe(f'rdap:{d}',rdap.lookup,d) or {};RDAP_DATA[d]=r
        for x in r.get('registrant_orgs',[]):observe(d,'domain','rdap:registrant_org',x)
        for x in r.get('registrar_orgs',[]):observe(d,'domain','rdap:registrar_org',x)
        for x in r.get('nameservers',[]):observe(d,'domain','rdap:nameserver',x)
else:status('rdap','SKIPPED')
display(pd.DataFrame([{'domain':d,'created':r.get('created'),'registrant_orgs':r.get('registrant_orgs'),'registrar_orgs':r.get('registrar_orgs')} for d,r in RDAP_DATA.items()]))

In [ ]:
#@title 12) Certificate Transparency
CT_DATA={}
if ENABLE_CT:
    for d in ROOTS:
        rows=safe(f'crt.sh:{d}',crtsh.lookup,d) or [];CT_DATA[d]=rows
        names=set()
        for row in rows if isinstance(rows,list) else []:
            for n in str(row.get('name_value','')).splitlines():
                n=n.strip().lower().lstrip('*.')
                if n:names.add(n)
        for n in list(names)[:1500]:observe(d,'domain','crt.sh',n)
else:status('crt.sh','SKIPPED')
print('CT rows:',sum(len(x) for x in CT_DATA.values()))

In [ ]:
#@title 13) Wayback Machine
WAYBACK_ROWS=[]
if ENABLE_WAYBACK:
    for d in ROOTS:
        rows=safe(f'wayback:{d}',history.wayback,d,1500) or []
        for r in rows[:1500]:
            WAYBACK_ROWS.append({'domain':d,'row':r})
            if r:observe(d,'domain','wayback',str(r[0]),notes=str(r[1]) if len(r)>1 else '')
else:status('wayback','SKIPPED')
print('Wayback:',len(WAYBACK_ROWS))

In [ ]:
#@title 14) Common Crawl
COMMONCRAWL_ROWS=[]
if ENABLE_COMMONCRAWL:
    for d in ROOTS:
        rows=safe(f'commoncrawl:{d}',history.commoncrawl,d,800) or []
        for r in rows[:800]:
            COMMONCRAWL_ROWS.append({'domain':d,**r})
            if r.get('url'):observe(d,'domain','commoncrawl',r['url'])
else:status('commoncrawl','SKIPPED')
print('Common Crawl:',len(COMMONCRAWL_ROWS))

In [ ]:
#@title 15) urlscan search
URLSCAN_ROWS=[]
if ENABLE_URLSCAN:
    for d in ROOTS:
        data=safe(f'urlscan:{d}',urlscan.search,d,SECRETS.get('URLSCAN_API_KEY','')) or {}
        rows=data.get('results',[]) if isinstance(data,dict) else []
        for r in rows:
            r['_root']=d;URLSCAN_ROWS.append(r)
            u=(r.get('page') or {}).get('url') or (r.get('task') or {}).get('url')
            if u:observe(d,'domain','urlscan',u)
else:status('urlscan','SKIPPED')
print('urlscan results:',len(URLSCAN_ROWS))

In [ ]:
#@title 16) urlscan Result API: redirects, resources e durable identifiers
URLSCAN_DETAILS=[];DURABLE_IDENTIFIERS=[];WEB_FINGERPRINTS=[]
if ENABLE_URLSCAN and ENABLE_URLSCAN_DETAILS:
    for r in URLSCAN_ROWS[:URLSCAN_DETAIL_MAX]:
        sid=(r.get('_id') or r.get('task',{}).get('uuid') or r.get('task',{}).get('id'))
        if not sid:continue
        detail=safe(f'urlscan-result:{sid}',urlscan.result,sid,SECRETS.get('URLSCAN_API_KEY',''))
        if not detail:continue
        URLSCAN_DETAILS.append(detail);ex=extract_durable_identifiers(detail);fp=passive_web_fingerprint(detail);WEB_FINGERPRINTS.append(fp)
        root=r.get('_root') or ''
        for ident in ex['identifiers']:
            row={'domain':root,**ident,'source':'urlscan'};DURABLE_IDENTIFIERS.append(row);observe(root,'domain',f"urlscan:identifier:{ident['type']}",ident['value'])
        for redir in ex['redirects']:observe(root,'domain','urlscan:redirect',redir)
print('Detailed scans:',len(URLSCAN_DETAILS),'| durable identifiers:',len(DURABLE_IDENTIFIERS))

In [ ]:
#@title 17) OTX / VirusTotal / ThreatFox
OTX_ROWS=[];VT_ROWS=[];THREATFOX_ROWS=[]
for d in ROOTS:
    if ENABLE_OTX:
        x=safe(f'otx:{d}',threatintel.otx_domain,d) or {};rows=x.get('url_list',[]) if isinstance(x,dict) else [];OTX_ROWS+=rows
        for r in rows[:300]:
            if isinstance(r,dict) and r.get('url'):observe(d,'domain','otx',r['url'])
    if ENABLE_VT:
        if SECRETS.get('VT_API_KEY'):
            x=safe(f'virustotal:{d}',threatintel.virustotal_domain,d,SECRETS['VT_API_KEY']);VT_ROWS.append(x) if x else None
        else:status('virustotal','SKIPPED',error='missing VT_API_KEY')
    if ENABLE_THREATFOX:
        if SECRETS.get('THREATFOX_AUTH_KEY'):
            x=safe(f'threatfox:{d}',threatintel.threatfox_search,d,SECRETS['THREATFOX_AUTH_KEY']);THREATFOX_ROWS.append(x) if x else None
        else:status('threatfox','SKIPPED',error='missing THREATFOX_AUTH_KEY')
print('OTX:',len(OTX_ROWS),'VT:',len(VT_ROWS),'ThreatFox:',len(THREATFOX_ROWS))

## D · Optional infrastructure intelligence

In [ ]:
#@title 18) DNSDumpster
DNSDUMPSTER_DATA={}
if ENABLE_DNSDUMPSTER:
    if not SECRETS.get('DNSDUMPSTER_API_KEY'):status('dnsdumpster','SKIPPED',error='missing DNSDUMPSTER_API_KEY')
    else:
        for d in ROOTS:
            raw=safe(f'dnsdumpster:{d}',dnsdumpster_lookup,d,SECRETS['DNSDUMPSTER_API_KEY'])
            if raw:DNSDUMPSTER_DATA[d]=dnsdumpster_normalize(d,raw)
print('DNSDumpster roots:',len(DNSDUMPSTER_DATA))

In [ ]:
#@title 19) FOFA
FOFA_DATA={}
if ENABLE_FOFA:
    if not SECRETS.get('FOFA_API_KEY'):status('fofa','SKIPPED',error='missing FOFA_API_KEY')
    else:
        for d in ROOTS:
            x=safe(f'fofa:{d}',fofa_domain,d,SECRETS['FOFA_API_KEY'],100,False)
            if x:FOFA_DATA[d]=x
print('FOFA roots:',len(FOFA_DATA))

In [ ]:
#@title 20) Censys
CENSYS_DATA={}
if ENABLE_CENSYS:
    if not SECRETS.get('CENSYS_PAT'):status('censys','SKIPPED',error='missing CENSYS_PAT')
    else:
        for d in ROOTS:
            x=safe(f'censys:{d}',censys_domain,d,SECRETS['CENSYS_PAT'],SECRETS.get('CENSYS_ORG_ID'),50)
            if x:CENSYS_DATA[d]=x
print('Censys roots:',len(CENSYS_DATA))

## E · Discovery, similarity & fingerprinting

In [ ]:
#@title 21) dnstwist lookalike discovery
DNSTWIST_ROWS=[]
if ENABLE_DNSTWIST and (ROOTS or BRAND):
    ref=ROOTS[0] if ROOTS else f'{BRAND}.com'
    t=time.time()
    try:
        proc=subprocess.run(['dnstwist','--registered','--format','json',ref],capture_output=True,text=True,timeout=180)
        rows=json.loads(proc.stdout) if proc.returncode==0 and proc.stdout.strip() else []
        for r in rows[:DNSTWIST_MAX]:
            dom=r.get('domain')
            if not dom:continue
            row={'domain':dom,'fuzzer':r.get('fuzzer'),'dns_a':r.get('dns_a',[]),'dns_mx':r.get('dns_mx',[]),'dns_ns':r.get('dns_ns',[])};DNSTWIST_ROWS.append(row);observe(dom,'domain','dnstwist',r.get('fuzzer',''),'derived')
        status('dnstwist','OK' if proc.returncode==0 else 'UNAVAILABLE',len(DNSTWIST_ROWS),proc.stderr,time.time()-t)
    except Exception as e:status('dnstwist','UNAVAILABLE',0,e,time.time()-t)
else:status('dnstwist','SKIPPED')
print('dnstwist candidates:',len(DNSTWIST_ROWS))

In [ ]:
#@title 22) Domain Similarity Engine
SIMILARITY_DOMAINS=sorted(set(DOMAINS+[x['domain'] for x in DNSTWIST_ROWS]))
DOMAIN_SIMILARITY=compare_domains(SIMILARITY_DOMAINS,min_score=.48,max_pairs=5000) if ENABLE_DOMAIN_SIMILARITY else []
for x in DOMAIN_SIMILARITY[:500]:
    if x['score']>=.72:observe(x['left'],'domain','derived:domain_similarity',f"{x['right']} score={x['score']}",'derived')
display(pd.DataFrame(DOMAIN_SIMILARITY[:100]))

In [ ]:
#@title 23) Passive Web / Kit Fingerprints
WEB_FP_ROWS=[]
for i,fp in enumerate(WEB_FINGERPRINTS):
    WEB_FP_ROWS.append({'scan_index':i,'sha256':fp.get('sha256'),'server':fp.get('server'),'title':fp.get('page_title'),'resource_hosts':fp.get('resource_hosts'),'identifiers':fp.get('identifiers')})
display(pd.DataFrame(WEB_FP_ROWS))

In [ ]:
#@title 24) Durable Identifier Index
DURABLE_INDEX=defaultdict(list)
for r in DURABLE_IDENTIFIERS:DURABLE_INDEX[(r['type'],r['value'])].append(r.get('domain'))
DURABLE_ROWS=[{'type':k[0],'value':k[1],'domains':sorted(set(v)),'domain_count':len(set(v))} for k,v in DURABLE_INDEX.items()]
display(pd.DataFrame(DURABLE_ROWS))

## F · Evidence engineering

In [ ]:
#@title 25) Evidence Ledger
LEDGER_DF=build_ledger(OBS,SETTINGS.source_reliability)
print('Evidence rows:',len(LEDGER_DF));display(LEDGER_DF.head(30))

In [ ]:
#@title 26) Source Independence Engine
SOURCE_INDEPENDENCE=source_independence(LEDGER_DF.to_dict('records') if not LEDGER_DF.empty else [])
display(pd.DataFrame(SOURCE_INDEPENDENCE))

In [ ]:
#@title 27) Negative Evidence
NEGATIVE_EVIDENCE=[]
for d in ROOTS:
    rd=RDAP_DATA.get(d,{})
    ctx={'rdap_checked':d in RDAP_DATA,'registrant_org_present':bool(rd.get('registrant_orgs')),'rdap_privacy_proxy':False,'historical_scan_expected':ENABLE_WAYBACK,'historical_scan_count':sum(1 for x in WAYBACK_ROWS if x['domain']==d),'expected_mx':False,'mx_present':bool(DNS_DATA.get(d,{}).get('MX'))}
    for row in evaluate_negative_evidence(ctx):NEGATIVE_EVIDENCE.append({'domain':d,**row})
display(pd.DataFrame(NEGATIVE_EVIDENCE))

## G · Relationship & campaign graph

In [ ]:
#@title 28) Relationship Graph 2.0
G=CampaignGraph()
for d in ROOTS:
    G.add_node(d,'domain')
    for rt,vals in DNS_DATA.get(d,{}).items():
        for v in vals:G.add_rel(d,str(v).lower().rstrip('.'),f'dns_{rt.lower()}',f'dns:{rt}',weight=.55 if rt in {'NS','MX','CNAME'} else .30)
    for org in RDAP_DATA.get(d,{}).get('registrant_orgs',[]):G.add_rel(d,'org:'+org,'registrant_org','rdap',weight=.92)
    for reg in RDAP_DATA.get(d,{}).get('registrar_orgs',[]):G.add_rel(d,'registrar:'+reg,'registered_by','rdap',weight=.18)
for r in DURABLE_ROWS:
    if r['domain_count']>1:
        node='tracker:'+r['type']+':'+r['value'];
        for d in r['domains']:G.add_rel(d,node,'same_tracker','urlscan',weight=.98)
for s in DOMAIN_SIMILARITY:
    if s['score']>=.72:G.add_rel(s['left'],s['right'],'lexical_similarity','derived',weight=s['score'])
print('nodes',G.g.number_of_nodes(),'edges',G.g.number_of_edges())

In [ ]:
#@title 29) Cluster Explosion Guard
GUARDED_COMPONENTS,G_GUARDED=guarded_components(G.g,min_weight=.45)
CAMPAIGN_CLUSTERS=[{'cluster_id':f'CAMP-{i:03d}','size':len(x),'nodes':sorted(map(str,x))[:250]} for i,x in enumerate(GUARDED_COMPONENTS,1) if len(x)>1]
display(pd.DataFrame(CAMPAIGN_CLUSTERS))

In [ ]:
#@title 30) Relationship export object
RELATIONSHIP_GRAPH={'nodes':[{'id':str(n),'type':d.get('node_type','entity')} for n,d in G_GUARDED.nodes(data=True)],'edges':[{'source':str(a),'target':str(b),'relations':d.get('relations',[]),'weight':d.get('weight',0),'sources':d.get('sources',[]),'evidence_count':d.get('evidence_count',1)} for a,b,d in G_GUARDED.edges(data=True)]}
print('export graph:',len(RELATIONSHIP_GRAPH['nodes']),'nodes /',len(RELATIONSHIP_GRAPH['edges']),'edges')

In [ ]:
#@title 31) Campaign Fingerprints
CAMPAIGN_FINGERPRINTS=[]
for c in CAMPAIGN_CLUSTERS:
    nodes=c['nodes'];cluster={'domains':[x for x in nodes if '.' in x and not x.startswith(('org:','registrar:','tracker:'))],'urls':[],'ips':[],'phones':[],'certificates':[]}
    fp=campaign_fingerprint(cluster);CAMPAIGN_FINGERPRINTS.append({'cluster_id':c['cluster_id'],**fp})
display(pd.DataFrame(CAMPAIGN_FINGERPRINTS))

In [ ]:
#@title 32) Operator Fingerprints
OPERATOR_FINGERPRINTS=[]
for c in CAMPAIGN_CLUSTERS:
    domains=[x for x in c['nodes'] if x in ROOTS];features={'registrars':[],'registrant_orgs':[],'nameservers':[],'mx':[],'certificates':[],'trackers':[],'jarm':[],'redirect_fingerprints':[],'template_hashes':[],'registration_hours_utc':[]}
    for d in domains:
        features['registrars']+=RDAP_DATA.get(d,{}).get('registrar_orgs',[]);features['registrant_orgs']+=RDAP_DATA.get(d,{}).get('registrant_orgs',[]);features['nameservers']+=DNS_DATA.get(d,{}).get('NS',[]);features['mx']+=DNS_DATA.get(d,{}).get('MX',[])
    for r in DURABLE_ROWS:
        if set(r['domains'])&set(domains):features['trackers'].append(r['type']+':'+r['value'])
    OPERATOR_FINGERPRINTS.append({'cluster_id':c['cluster_id'],**operator_fingerprint(features)})
display(pd.DataFrame(OPERATOR_FINGERPRINTS))

## H · Temporal & infrastructure strategy

In [ ]:
#@title 33) Timeline
TIMELINE=build_timeline(LEDGER_DF)
display(pd.DataFrame(TIMELINE).head(100))

In [ ]:
#@title 34) Campaign lifecycle
LIFECYCLE=infer_campaign_lifecycle([{'observed_at':o.observed_at} for o in OBS])
display(pd.DataFrame([LIFECYCLE]))

In [ ]:
#@title 35) Infrastructure churn
CHURN=infrastructure_churn([o.__dict__ for o in OBS]);display(pd.DataFrame([CHURN]))

## I · Ownership & attribution engineering

In [ ]:
#@title 36) Ownership normalization
OWNERSHIP={}
for d in ROOTS:
    x={'registrant_orgs':RDAP_DATA.get(d,{}).get('registrant_orgs',[]),'registrar_orgs':RDAP_DATA.get(d,{}).get('registrar_orgs',[]),'nameservers':DNS_DATA.get(d,{}).get('NS',[]),'mx':DNS_DATA.get(d,{}).get('MX',[]),'ips':DNS_DATA.get(d,{}).get('A',[]),'asns':[],'providers':[],'certificates':[],'jarm':[],'public_contact_refs':[]}
    dd=DNSDUMPSTER_DATA.get(d,{})
    x['asns']+=sorted({str(r.get('asn')) for r in dd.get('records',[]) if r.get('asn')});x['providers']+=dd.get('asn_owners',[])
    for row in FOFA_DATA.get(d,{}).get('rows',[]):
        if row.get('asn'):x['asns'].append(str(row['asn']))
        if row.get('org'):x['providers'].append(str(row['org']))
        if row.get('jarm'):x['jarm'].append(str(row['jarm']))
    OWNERSHIP[d]={k:sorted(set(v)) if isinstance(v,list) else v for k,v in x.items()}
display(pd.DataFrame([{'domain':d,**x} for d,x in OWNERSHIP.items()]))

In [ ]:
#@title 37) Operator Correlation + ACH
ATTRIBUTIONS=[];engine=AttributionEngine();roots=list(ROOTS)
for i,left in enumerate(roots):
    for right in roots[i+1:]:
        a,b=OWNERSHIP[left],OWNERSHIP[right];ev=[]
        def add(code,fam,source,detail,rel):ev.append({'code':code,'source_family':fam,'source':source,'detail':detail,'source_reliability':rel})
        def ov(k):return sorted(set(a.get(k,[]))&set(b.get(k,[])))
        if ov('registrant_orgs'):add('same_registrant_org','registration','rdap',ov('registrant_orgs'),.95)
        if ov('jarm'):add('same_jarm','service_fingerprint','fofa',ov('jarm'),.85)
        if ov('asns'):add('same_asn','infrastructure','internet_scan',ov('asns'),.85)
        if ov('providers'):add('same_provider','infrastructure','internet_scan',ov('providers'),.80)
        if ov('ips'):add('same_ip','infrastructure','dns',ov('ips'),.95)
        shared_tracker=[r for r in DURABLE_ROWS if left in r['domains'] and right in r['domains']]
        if shared_tracker:add('same_tracker','web_scan','urlscan',[x['value'] for x in shared_tracker],.90)
        if ev:ATTRIBUTIONS.append(engine.assess(left,right,ev))
display(pd.DataFrame(ATTRIBUTIONS))

In [ ]:
#@title 38) Attribution Ladder summary
ATTRIBUTION_LADDER=[{'left':a['left'],'right':a['right'],'level':a['attribution_level'],'operator_confidence':a['common_operator_confidence'],'identity':a['identity_attribution'],'known_actor':a['known_actor_attribution'],'independent_sources':a['independent_source_families'],'strong_evidence':a['strong_evidence_count']} for a in ATTRIBUTIONS]
display(pd.DataFrame(ATTRIBUTION_LADDER))

## J · Passive takeover exposure

In [ ]:
#@title 39) Dangling CNAME exposure detector
TAKEOVER_EXPOSURE=[]
if ENABLE_PASSIVE_TAKEOVER:
    for d in ROOTS:
        targets=DNS_DATA.get(d,{}).get('CNAME',[])
        TAKEOVER_EXPOSURE+=assess_passive_exposure(d,targets,dns.query)
display(pd.DataFrame(TAKEOVER_EXPOSURE))

## K · Victimology, objective & intelligence requirements

In [ ]:
#@title 40) Victimology
VICTIMOLOGY=infer_victimology(LURE_TEXT,BRAND,IOCS.get('url',[]),[(r.get('page') or {}).get('title','') for r in URLSCAN_ROWS]);display(pd.DataFrame([VICTIMOLOGY]))

In [ ]:
#@title 41) Adversary objective
OBJECTIVES=infer_objectives(LURE_TEXT,IOCS.get('url',[]));display(pd.DataFrame([OBJECTIVES]))

In [ ]:
#@title 42) PIRs / Intelligence Requirements
PIRS=[x.to_dict() for x in default_phishing_requirements()];display(pd.DataFrame(PIRS))

## L · IOC quality, decisioning & next collection

In [ ]:
#@title 43) Warning Lists / false-positive guard
WARNING_ENGINE=WarningListEngine()
WARNING_HITS=[]
for d in ROOTS:
    for ip in DNS_DATA.get(d,{}).get('A',[]):
        for hit in WARNING_ENGINE.check(ip,{}):WARNING_HITS.append({'ioc':ip,**hit})
display(pd.DataFrame(WARNING_HITS))

In [ ]:
#@title 44) IOC Decision Engine
recent_cutoff=datetime.now(timezone.utc)-timedelta(days=7);IOC_CANDIDATES=[]
for d in ROOTS:
    rows=[o for o in OBS if o.entity==d];families=len(set(o.source.split(':')[0] for o in rows));recent=False
    for o in rows:
        try:recent|=datetime.fromisoformat(o.observed_at.replace('Z','+00:00')).astimezone(timezone.utc)>=recent_cutoff
        except Exception:pass
    support=min(.80,.18+min(.32,len(rows)*.022)+min(.22,families*.055))
    if any((r.get('page',{}).get('domain') or '').lower()==d for r in URLSCAN_ROWS):support=min(.92,support+.10)
    IOC_CANDIDATES.append({'value':d,'type':'domain','confidence':support,'active':recent,'evidence_count':len(rows),'source_families':families,'context':{},'campaign':CASE_ID,'rationale':['passive evidence aggregation']})
IOC_DECISIONS=build_ioc_decisions(IOC_CANDIDATES,WARNING_ENGINE);display(pd.DataFrame(IOC_DECISIONS))

In [ ]:
#@title 45) Collection Gap Engine
families=len(SOURCE_INDEPENDENCE)
ctx={'active_state_known':any(x['active_now'] for x in IOC_DECISIONS),'registration_entity_known':any(x.get('registrant_orgs') for x in OWNERSHIP.values()),'redirect_chain_known':any(o.source=='urlscan:redirect' for o in OBS),'credential_endpoint_known':False,'victimology_known':VICTIMOLOGY.get('confidence') not in {'LOW','INSUFFICIENT'},'operator_identity_known':any(a.get('identity_attribution')=='SUPPORTED' for a in ATTRIBUTIONS),'independent_source_families':families}
COLLECTION_GAPS=build_collection_gaps(ctx);display(pd.DataFrame(COLLECTION_GAPS))

In [ ]:
#@title 46) Next Best Pivot Engine
PIVOT_CANDIDATES=[]
for r in DURABLE_ROWS:PIVOT_CANDIDATES.append({'value':r['value'],'type':'tracker','source':'urlscan','result_count':r['domain_count'],'evidence_value':.9,'novelty':.9,'source_reliability':.9,'expected_noise':.18})
for d,x in OWNERSHIP.items():
    for o in x.get('registrant_orgs',[]):PIVOT_CANDIDATES.append({'value':o,'type':'public_contact','source':'rdap','result_count':3,'evidence_value':.9,'novelty':.8,'source_reliability':.95,'expected_noise':.25})
    for j in x.get('jarm',[]):PIVOT_CANDIDATES.append({'value':j,'type':'jarm','source':'fofa','result_count':8,'evidence_value':.75,'novelty':.85,'source_reliability':.85,'expected_noise':.28})
    for a in x.get('asns',[]):PIVOT_CANDIDATES.append({'value':a,'type':'asn','source':'internet_scan','result_count':1500,'evidence_value':.45,'novelty':.4,'source_reliability':.85,'expected_noise':.92})
PIVOTS=rank_pivots(PIVOT_CANDIDATES);display(pd.DataFrame(PIVOTS[:50]))

In [ ]:
#@title 47) Multidimensional Confidence
max_op=max([x.get('common_operator_score',0) for x in ATTRIBUTIONS] or [0]);max_ioc=max([x.get('confidence',0) for x in IOC_DECISIONS] or [0])
CONFIDENCE=multidimensional_confidence(infrastructure_relationship=.85 if G_GUARDED.number_of_edges() else .25,common_operator=max_op,malicious_intent=max_ioc,campaign_active=.75 if LIFECYCLE.get('stage') in {'ACTIVE','EXPANSION','REACTIVATED'} else .3,victimology=.65 if VICTIMOLOGY.get('confidence')=='MODERATE' else .3,identity_attribution=.85 if any(a.get('identity_attribution')=='SUPPORTED' for a in ATTRIBUTIONS) else 0)
display(pd.DataFrame(CONFIDENCE).T)

In [ ]:
#@title 48) Action Matrix
DETECTION_PACKAGE=build_detection_package(IOC_DECISIONS,BRAND)
ACTION_MATRIX=build_action_matrix(IOC_DECISIONS,COLLECTION_GAPS,DETECTION_PACKAGE);display(pd.DataFrame(ACTION_MATRIX))

## M · Detection engineering

In [ ]:
#@title 49) Detection Bridge: Sigma / Splunk / Sentinel / Elastic / Wazuh / Suricata
print(json.dumps(DETECTION_PACKAGE,ensure_ascii=False,indent=2)[:20000])

## N · Executive intelligence

In [ ]:
#@title 50) Executive Assessment
EXECUTIVE_ASSESSMENT={'judgment':f'Case {CASE_ID}: {LIFECYCLE.get("stage")} · objective {OBJECTIVES.get("primary")} · {sum(1 for x in IOC_DECISIONS if x.get("block_recommended"))} IOC(s) currently meet the conservative block policy.','implication':'Prioritize P1 actions, preserve evidence, and close collection gaps before stronger attribution. Shared infrastructure remains weak evidence.'}
INTEL_BRIEF=build_intelligence_brief(CASE_ID,EXECUTIVE_ASSESSMENT,ACTION_MATRIX,COLLECTION_GAPS,PIVOTS,LIFECYCLE,CONFIDENCE,VICTIMOLOGY,OBJECTIVES,changes={'infrastructure_churn':CHURN})
print(json.dumps(INTEL_BRIEF,ensure_ascii=False,indent=2)[:16000])

In [ ]:
#@title 51) Feature coverage / validação do caso
FEATURE_MATRIX=[
 {'feature':'multi_ioc_ingestion','status':'ACTIVE','evidence':len(INVENTORY)},
 {'feature':'dns_rdap_ct','status':'ACTIVE','evidence':len(DNS_DATA)+len(RDAP_DATA)+len(CT_DATA)},
 {'feature':'historical_web','status':'ACTIVE' if ENABLE_WAYBACK or ENABLE_COMMONCRAWL else 'DISABLED','evidence':len(WAYBACK_ROWS)+len(COMMONCRAWL_ROWS)},
 {'feature':'domain_similarity','status':'ACTIVE' if ENABLE_DOMAIN_SIMILARITY else 'DISABLED','evidence':len(DOMAIN_SIMILARITY)},
 {'feature':'durable_identifiers','status':'ACTIVE','evidence':len(DURABLE_ROWS)},
 {'feature':'source_independence','status':'ACTIVE','evidence':len(SOURCE_INDEPENDENCE)},
 {'feature':'cluster_guard','status':'ACTIVE','evidence':len(CAMPAIGN_CLUSTERS)},
 {'feature':'negative_evidence','status':'ACTIVE','evidence':len(NEGATIVE_EVIDENCE)},
 {'feature':'attribution_ach','status':'ACTIVE','evidence':len(ATTRIBUTIONS)},
 {'feature':'passive_takeover','status':'ACTIVE' if ENABLE_PASSIVE_TAKEOVER else 'DISABLED','evidence':len(TAKEOVER_EXPOSURE)},
 {'feature':'ioc_decisioning','status':'ACTIVE','evidence':len(IOC_DECISIONS)},
 {'feature':'report_backend','status':'READY','evidence':0},
]
display(pd.DataFrame(FEATURE_MATRIX))

## O · Investigation report backend

In [ ]:
#@title 52) Montar REPORT_DATA completo
REPORT_DATA={'meta':{'case_id':CASE_ID,'analyst':ANALYST,'brand':BRAND,'generated_at':now_iso(),'tool':'Tropeiro Intel','version':'4.0.0','mode':MODE},'executive_brief':INTEL_BRIEF,'executive_assessment':EXECUTIVE_ASSESSMENT,'action_matrix':ACTION_MATRIX,'ioc_decisions':IOC_DECISIONS,'collection_gaps':COLLECTION_GAPS,'next_best_pivots':PIVOTS,'confidence':CONFIDENCE,'victimology':VICTIMOLOGY,'objectives':OBJECTIVES,'lifecycle':LIFECYCLE,'infrastructure_churn':CHURN,'attribution_assessments':ATTRIBUTIONS,'attribution_ladder':ATTRIBUTION_LADDER,'campaign_clusters':CAMPAIGN_CLUSTERS,'campaign_fingerprints':CAMPAIGN_FINGERPRINTS,'operator_fingerprints':OPERATOR_FINGERPRINTS,'evidence_ledger':LEDGER_DF.to_dict('records') if not LEDGER_DF.empty else [],'source_status':STATUS,'source_independence':SOURCE_INDEPENDENCE,'provider_plan':PROVIDER_PLAN,'ownership':OWNERSHIP,'relationship_graph':RELATIONSHIP_GRAPH,'domain_similarity':DOMAIN_SIMILARITY,'durable_identifiers':DURABLE_ROWS,'web_fingerprints':WEB_FP_ROWS,'negative_evidence':NEGATIVE_EVIDENCE,'timeline':TIMELINE,'takeover_exposure':TAKEOVER_EXPOSURE,'detection_package':DETECTION_PACKAGE,'feature_matrix':FEATURE_MATRIX}
print('REPORT_DATA sections:',len(REPORT_DATA))

In [ ]:
#@title 53) Gerar relatório enriquecido local/offline
REPORT_DIR=WORKSPACE/'report';REPORT_DIR.mkdir(parents=True,exist_ok=True);REPORT_PATH=REPORT_DIR/'Tropeiro_Intel_Investigation_Report.html'
build_report(REPORT_DATA,REPORT_PATH);print('Report:',REPORT_PATH)

In [ ]:
#@title 54) Visualizar relatório dentro do Colab
if REPORT_PATH.exists():display(IFrame(src=str(REPORT_PATH),width='100%',height=1050))

## P · Export center & case package

In [ ]:
#@title 55) Export Center seletivo
import ipywidgets as widgets
from IPython.display import clear_output
choices={'Relatório HTML':'report_html','Executive Brief JSON':'executive_brief_json','IOC Decisions CSV':'ioc_decisions_csv','Action Matrix CSV':'action_matrix_csv','Collection Gaps CSV':'collection_gaps_csv','Next Best Pivots CSV':'next_best_pivots_csv','Evidence Ledger CSV':'evidence_ledger_csv','Attribution CSV':'attribution_csv','Domain Similarity CSV':'domain_similarity_csv','Durable Identifiers CSV':'durable_identifiers_csv','Negative Evidence CSV':'negative_evidence_csv','Takeover Exposure CSV':'takeover_exposure_csv','Source Independence CSV':'source_independence_csv','Provider Plan CSV':'provider_plan_csv','Timeline CSV':'timeline_csv','Relationship Graph JSON':'relationship_graph_json','Campaign Fingerprints JSON':'campaign_fingerprints_json','Operator Fingerprints JSON':'operator_fingerprints_json','Detection Package JSON':'detection_json','Full Case JSON':'case_json'}
checks={label:widgets.Checkbox(value=(key in {'report_html','executive_brief_json','ioc_decisions_csv','action_matrix_csv'}),description=label) for label,key in choices.items()};btn=widgets.Button(description='Gerar pacote selecionado');out=widgets.Output()
def _export(_):
    with out:
        clear_output();selected={choices[label] for label,w in checks.items() if w.value};files=export_selected(REPORT_DATA,WORKSPACE/'export',selected,REPORT_PATH);zp=zip_exports(files,WORKSPACE/f'Tropeiro_{CASE_ID}_Export.zip');print('ZIP:',zp)
        try:
            from google.colab import files as colab_files;colab_files.download(str(zp))
        except Exception:pass
btn.on_click(_export);display(widgets.VBox(list(checks.values())+[btn,out]))

In [ ]:
#@title 56) Snapshot do caso + checksums
snapshot=WORKSPACE/'case_snapshot.json';snapshot.write_text(json.dumps(REPORT_DATA,ensure_ascii=False,indent=2,default=str),encoding='utf-8')
manifest=[]
for p in [snapshot,REPORT_PATH]:
    if p.exists():manifest.append({'file':p.name,'bytes':p.stat().st_size,'sha256':hashlib.sha256(p.read_bytes()).hexdigest()})
print(json.dumps(manifest,indent=2))

In [ ]:
#@title 57) Source health final
STATUS_DF=pd.DataFrame(STATUS)
display(STATUS_DF if not STATUS_DF.empty else pd.DataFrame([{'status':'no collector status'}]))

## Encerramento analítico

- **Similarity ≠ maliciousness.**
- **Same kit ≠ same operator.**
- **Same operator ≠ real-world identity.**
- **Shared hosting/CDN ≠ campaign ownership.**
- **No hit ≠ benign.**
- **Passive takeover signal ≠ autorização para reivindicar recurso.**
- Preserve `Evidence Ledger`, `Source Health`, `Feature Matrix`, relatório e checksums com o caso.